In [33]:
import pandas as pd
import numpy as np

# pip install category-encoders
import category_encoders as ce

In [34]:
# it seems the data doesn't follow the UTF-8 format
# let's force the encoding to follow Latin1 (or ISO-8859-1)
df = pd.read_csv("../OnlineRetail.csv", encoding="ISO-8859-1")

df['Sales'] = df['Quantity'] * df['UnitPrice']
df


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France,16.60


In [35]:
# right, we seem to have over 4000 unique options
df['Description'].nunique()

4223

**Example 1: Target encoding (replace each unique category with that category's average)**

In [36]:
# create a target encoder by using category encoders
encoder = ce.TargetEncoder(cols=["Description"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'], df['Sales'])

# insert the encoded description averages back to the original data
df['Description_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510


In [37]:
# based on target encoding, average price for WHITE METAL LANTERN
df_test = df[df['Description'] == "WHITE METAL LANTERN"]
df_test['Sales'].mean().item()

# Sales could be our target variable
# for each category in description
# we replace the text category with its average of the sales value

24.13643292682927

**If data overfits easily with ML model and the target encoded data (average), you can use LeaveOneOutEncoder instead to reduce overfitting (adds a bit of variance)**

In [38]:
# create a target encoder by using category encoders
encoder = ce.LeaveOneOutEncoder(cols=["Description"])

# apply the encoder, Description is going to be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'], df['Sales'])

# insert the encoded description averages back to the original data
df['Description_LOO_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded,Description_LOO_encoded
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959,42.083264
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433,24.148043
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590,25.591849
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825,34.079873
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510,48.812924


In [ ]:
# Leave One Out-encoder is almost the same as 
# Target Encoding, BUT for each row in the dataset
# and the average target calculation, THAT ONE PARTICULAR
# row is excluded from the calculation
category_row_sales = {
    2,
    5,
    #7,
    2,
    5,
    1
}

# quick average calculation
average = sum(category_row_sales) / len(category_row_sales)
average

2.6666666666666665

In [39]:
df['Description'].value_counts().head(5)


Description
WHITE HANGING HEART T-LIGHT HOLDER    2369
REGENCY CAKESTAND 3 TIER              2200
JUMBO BAG RED RETROSPOT               2159
PARTY BUNTING                         1727
LUNCH BAG RED RETROSPOT               1638
Name: count, dtype: int64